# Part 7 · requests / API
> requests / httpx / urllib / 认证 / 分页 / 速率限制 / Webhook

## 1. requests 基础

In [ ]:
import requests

# --- GET ---
resp = requests.get('https://api.example.com/orders')

# 带查询参数
resp = requests.get(
    'https://api.example.com/orders',
    params={'status': 'shipped', 'limit': 100, 'page': 1},
    # 自动拼成: ?status=shipped&limit=100&page=1
    headers={'Authorization': 'Bearer TOKEN123', 'Accept': 'application/json'},
    timeout=(5, 30),    # (连接超时, 读取超时) 秒
    verify=True,        # SSL 验证（False 跳过，不推荐生产用）
)

# --- 响应处理 ---
resp.status_code          # 200
resp.ok                   # True if status_code < 400
resp.reason               # 'OK'
resp.text                 # 响应体字符串
resp.content              # 响应体 bytes
resp.json()               # 解析 JSON → dict/list
resp.headers              # 响应头（字典）
resp.headers['Content-Type']
resp.url                  # 最终 URL（含重定向）
resp.elapsed              # 请求耗时（timedelta）
resp.encoding             # 字符编码

# 非 2xx 自动抛异常
resp.raise_for_status()   # 4xx/5xx → requests.exceptions.HTTPError

# --- POST ---
resp = requests.post(
    'https://api.example.com/orders',
    json={'item': 'book', 'qty': 3},   # JSON body（自动设 Content-Type）
    headers={'Authorization': 'Bearer TOKEN'},
)

# Form 数据
resp = requests.post(url, data={'field1': 'val1'})  # application/x-www-form-urlencoded

# 上传文件
with open('data.csv', 'rb') as f:
    resp = requests.post(url, files={'file': ('data.csv', f, 'text/csv')})

# --- PUT / PATCH / DELETE ---
requests.put(url, json=payload)
requests.patch(url, json={'status': 'cancelled'})
requests.delete(url, headers=headers)

## 2. Session — 复用连接 & 共享配置

In [ ]:
import requests

# Session 复用 TCP 连接，多次请求时比 requests.get() 快
with requests.Session() as session:
    # 共享配置（只设一次）
    session.headers.update({
        'Authorization': 'Bearer TOKEN123',
        'Accept': 'application/json',
    })
    session.verify = True

    # 所有请求自动带上共享 headers
    r1 = session.get('https://api.example.com/orders')
    r2 = session.get('https://api.example.com/customers')

# --- 认证方式 ---
# Basic Auth
session.auth = ('username', 'password')
requests.get(url, auth=('user', 'pass'))

# Bearer Token
session.headers['Authorization'] = f'Bearer {token}'

# API Key
session.headers['X-API-Key'] = api_key
requests.get(url, params={'api_key': api_key})

# OAuth2（requests-oauthlib）
from requests_oauthlib import OAuth2Session
oauth = OAuth2Session(client_id, token=token)
resp = oauth.get(url)

# --- 重试机制 ---
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

retry_strategy = Retry(
    total=3,                          # 最多重试3次
    backoff_factor=1,                 # 重试间隔：1, 2, 4 秒
    status_forcelist=[429, 500, 502, 503, 504],  # 这些状态码重试
    allowed_methods=['GET', 'POST'],  # 允许重试的方法
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session = requests.Session()
session.mount('https://', adapter)
session.mount('http://', adapter)

## 3. 分页处理 & 速率限制

In [ ]:
import requests, time

# --- 分页：offset/limit 模式 ---
def fetch_all_offset(base_url, headers, page_size=100):
    all_data = []
    offset = 0
    while True:
        resp = requests.get(base_url,
                            params={'limit': page_size, 'offset': offset},
                            headers=headers)
        resp.raise_for_status()
        data = resp.json().get('data', [])
        if not data:
            break
        all_data.extend(data)
        offset += page_size
    return all_data

# --- 分页：cursor/next_url 模式 ---
def fetch_all_cursor(base_url, headers):
    all_data = []
    url = base_url
    while url:
        resp = requests.get(url, headers=headers)
        resp.raise_for_status()
        body = resp.json()
        all_data.extend(body['results'])
        url = body.get('next')   # None 时退出
    return all_data

# --- 分页：page 编号 ---
def fetch_all_pages(base_url, headers):
    all_data = []
    page = 1
    while True:
        resp = requests.get(base_url, params={'page': page}, headers=headers)
        resp.raise_for_status()
        body = resp.json()
        all_data.extend(body['items'])
        if page >= body['total_pages']:
            break
        page += 1
    return all_data

# --- 速率限制处理 ---
def request_with_retry(url, headers, max_retries=5):
    for attempt in range(max_retries):
        resp = requests.get(url, headers=headers)
        if resp.status_code == 429:   # Too Many Requests
            wait = int(resp.headers.get('Retry-After', 2 ** attempt))
            print(f'Rate limited, waiting {wait}s')
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp.json()
    raise Exception('Max retries exceeded')

# --- 限速（控制请求频率）---
import time
from ratelimit import limits, sleep_and_retry  # pip install ratelimit

@sleep_and_retry
@limits(calls=10, period=1)    # 每秒最多10次
def api_call(url):
    return requests.get(url)

## 4. 错误处理

In [ ]:
import requests
from requests.exceptions import (
    ConnectionError,    # 网络不通
    Timeout,            # 超时
    HTTPError,          # 4xx/5xx（raise_for_status 触发）
    RequestException,   # 所有 requests 异常的基类
)

try:
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()     # 4xx/5xx 抛 HTTPError
    data = resp.json()
except Timeout:
    print('请求超时')
except ConnectionError:
    print('网络连接失败')
except requests.exceptions.HTTPError as e:
    print(f'HTTP 错误: {e.response.status_code}')
    if e.response.status_code == 401:
        print('未授权，请检查 Token')
    elif e.response.status_code == 404:
        print('资源不存在')
    elif e.response.status_code == 422:
        print('请求参数错误:', e.response.json())
except RequestException as e:
    print(f'请求失败: {e}')

## 5. urllib.parse — URL 处理

In [ ]:
from urllib.parse import urlparse, urljoin, urlencode, quote, unquote, parse_qs, parse_qsl

# URL 解析
parsed = urlparse('https://api.example.com/v1/orders?status=shipped&limit=10#section1')
parsed.scheme      # 'https'
parsed.netloc      # 'api.example.com'
parsed.path        # '/v1/orders'
parsed.query       # 'status=shipped&limit=10'
parsed.fragment    # 'section1'

# 查询参数解析
parse_qs('status=shipped&limit=10')      # {'status':['shipped'],'limit':['10']}
parse_qsl('status=shipped&limit=10')     # [('status','shipped'),('limit','10')]

# URL 编码
urlencode({'status': 'pending review', 'limit': 10})  # 'status=pending+review&limit=10'
quote('hello world')                      # 'hello%20world'
quote('https://a.com/b', safe='/:')       # safe 里的字符不编码
unquote('hello%20world')                  # 'hello world'

# URL 拼接
urljoin('https://api.example.com/v1/', 'orders')   # 'https://api.example.com/v1/orders'
urljoin('https://api.example.com/v1/orders', '/users')  # 'https://api.example.com/users'

# 动态构建 URL
base = 'https://api.example.com/orders'
params = urlencode({'status': 'shipped', 'limit': 100})
full_url = f'{base}?{params}'

## 6. httpx — 异步 HTTP（高并发场景）

In [ ]:
import httpx
import asyncio

# --- 同步（requests 替代品）---
resp = httpx.get('https://api.example.com/data', timeout=10)
resp.json()

# --- 异步（并发多个请求）---
async def fetch_all(urls):
    async with httpx.AsyncClient(timeout=30.0) as client:
        tasks = [client.get(url) for url in urls]
        responses = await asyncio.gather(*tasks)  # 并发发出所有请求
        return [r.json() for r in responses]

# 运行
urls = [f'https://api.example.com/orders/{i}' for i in range(100)]
results = asyncio.run(fetch_all(urls))   # 串行需要 100s，并发约 2-5s

# --- 控制并发数 ---
import asyncio

async def fetch_with_limit(urls, max_concurrent=10):
    sem = asyncio.Semaphore(max_concurrent)  # 最多 10 个并发
    async with httpx.AsyncClient() as client:
        async def fetch(url):
            async with sem:
                return await client.get(url)
        return await asyncio.gather(*[fetch(u) for u in urls])